# Sanity Check — CAPE (PF) Forward Pass

Goal: confirm the model loads, a single batch produces the right output shapes (`14x14xC` class activation maps for 448x448 input), and the distillation loss is finite and decreases when we overfit a handful of gradient steps on one batch.

**Do not run full training until every cell here passes.**

Assumes:
- `../../cape-reference` contains the cloned `AIML-MED/CAPE` repo (adjust `CAPE_REPO_PATH` below if different)
- `CUB_200_2011.tgz` has been repacked and sits in `DATA_ROOT` (see `data/README.md`)

In [1]:
import sys, os

CAPE_REPO_PATH = os.path.expanduser('~/cape-reference')   # adjust if your clone lives elsewhere
DATA_ROOT = os.path.expanduser('~/cape-reproduction/data')  # folder that directly CONTAINS CUB_200_2011.tgz

sys.path.insert(0, CAPE_REPO_PATH)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from models.model import Net
from models.losses import SoftTargetKDLoss
from datasets import CUB200
from configs import transforms as cape_transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


In [2]:
# Config mirrors configs/cub/resnet50_PF.py, with data_path overridden to our own machine.
# NOTE: we did not read the full main.py, so this dict is assembled from what the config
# file + Net.__init__ + Trainer.__init__ actually consume, not copy-pasted from a source we saw whole.
config = {
    'num_epochs': 30,
    'image_size': 448,
    'batch_size': 4,          # small for the sanity check, not the real 32
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'T_kld': 2,
    'settings': 'PF',
    'network': 'resnet50',
    'pretrained': True,
    'reload_path': os.path.join(CAPE_REPO_PATH, 'saved_models', 'cub_resnet50_TS.pth'),
    'loss_alpha': 0,
    'loss_beta': 1,
    'num_class': 200,
    'device': device,
}
config['cls_cols_dict'] = {'orig': config['num_class'], 'cape': config['num_class']}

In [3]:
# Build model and load the authors' TS checkpoint as the PF starting point
net = Net(config).to(device)

state_dict = torch.load(config['reload_path'], map_location=device)
net.load_state_dict(state_dict)
print('Loaded TS checkpoint from', config['reload_path'])

Loaded TS checkpoint from C:\Users\umama/cape-reference\saved_models\cub_resnet50_TS.pth


In [4]:
# Freeze backbone and the vanilla ('orig') classifier — PF only trains the 'cape'
# classifier plus the three Temperature modules (t1, t2, t3).
for p in net.net.parameters():
    p.requires_grad = False
for p in net.classifiers['orig'].parameters():
    p.requires_grad = False

trainable_params = [p for p in net.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable_params)
n_total = sum(p.numel() for p in net.parameters())
print(f'Trainable params: {n_trainable:,} / {n_total:,} total')

# BatchNorm layers inside the backbone stay in eval mode even during .train(),
# matching trainer.py's PF branch
net.train()
net.net.eval()

Trainable params: 409,803 / 24,327,635 total


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, ker

In [ ]:
# One small batch from the real CUB200 loader
train_transform, _ = cape_transforms.cub_transform(config['image_size'])
train_dataset = CUB200(root=DATA_ROOT, train=True, transform=train_transform)
loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, num_workers=0)

images, targets = next(iter(loader))
images = images.to(device)
targets = targets.type(torch.LongTensor).to(device)
print('Batch images shape:', images.shape)
print('Batch targets shape:', targets.shape)

# Forward pass — check the class activation map shape before pooling
with torch.no_grad():
    outputs = net(images)

cape_map = outputs['cape']['cls_map']
orig_map = outputs['orig']['cls_map']
cape_outcome = outputs['cape']['outcome']

print('cape cls_map shape:', cape_map.shape, '(expect batch x 200 x 14 x 14 for 448 input)')
print('orig cls_map shape:', orig_map.shape)
print('cape pooled outcome shape:', cape_outcome.shape, '(expect batch x 200)')

assert cape_map.shape[-2:] == (14, 14), 'Spatial size is not 14x14 — check image_size / backbone stride'
assert cape_outcome.shape[1] == config['num_class'], 'Pooled output does not match num_class'
print('Shape checks passed.')

Train file has been extracted


In [ ]:
# Overfit this single batch for a few steps — loss should be finite and trend downward.
# This mirrors trainer.py's train() loss exactly: loss_alpha=0 so class_loss is always 0,
# only the KD loss between CAPE's soft outcome and the frozen orig classifier's outcome drives training.
kl_loss_fn = SoftTargetKDLoss(T=config['T_kld']).to(device)
classification_loss_fn = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.SGD(trainable_params, lr=config['learning_rate'], weight_decay=config['weight_decay'])

losses = []
for step in range(10):
    optimizer.zero_grad()
    outputs = net(images)

    class_loss = config['loss_alpha'] * classification_loss_fn(outputs['orig']['outcome'], targets)
    kld_loss = config['loss_beta'] * kl_loss_fn(outputs['cape']['outcome_soft'], outputs['orig']['outcome'].detach())
    loss = class_loss + kld_loss

    assert torch.isfinite(loss).all(), f'Non-finite loss at step {step}: {loss.item()}'

    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    print(f'step {step}: loss={loss.item():.6f}')

print('\nAll losses finite:', all(l == l and abs(l) != float('inf') for l in losses))
print('Loss decreased overall:', losses[-1] < losses[0])

In [ ]:
import matplotlib.pyplot as plt

plt.plot(losses, marker='o')
plt.xlabel('Step')
plt.ylabel('KLD loss')
plt.title('Sanity check: single-batch overfit')
plt.show()

## If this all passed:
- Shapes are correct (14x14 CAM, batch x 200 pooled output)
- Loss stayed finite across steps
- Loss trended down while overfitting one batch

You're clear to move on to the small-subset training run, then the full PF training run.